# Hafta 2 · Kuantum Programlama İçin Gereken Matematik
**Ders:** Kuantum Hesaplama ve Uygulamaları · **Lab süresi:** ~50 dk · **Ortam:** Google Colab

Bu hafta matematiği **sadece kod yazmak için gerektiği kadar** işliyoruz ve her kavramı NumPy ile deniyoruz. Haftanın sonunda **kendi çok kübitli kuantum simülatörümüzü** yazıp sonuçlarını Qiskit ile karşılaştıracağız.

| Bölüm | Konu | Süre |
|---|---|---|
| 0 | Kurulum ve yardımcı fonksiyonlar | 3 dk |
| A | Karmaşık sayılar: büyüklük, eşlenik, faz | 7 dk |
| B | Vektörler ve iç çarpım: olasılığın genel formülü | 7 dk |
| C | Matrisler = kapılar: çarpma, sıra, üniterlik | 8 dk |
| D | Faz: global faz ve göreli faz (Bloch küresi) | 5 dk |
| E | Tensör çarpımı: kübitleri birleştirmek | 8 dk |
| F | **MiniSim**: kendi durum vektörü simülatörümüz + Qiskit ile doğrulama | 12 dk |
| G | Beklenen değer ⟨Z⟩ | ödev |
| H | Alıştırmalar | ödev |

## 0 · Kurulum

In [ ]:
!pip install -q qiskit qiskit-aer pylatexenc

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from qiskit.quantum_info import Statevector

np.set_printoptions(precision=3, suppress=True)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
NAVY, BLUE, ORANGE, GRAY = "#1F3A5F", "#2E6DB4", "#D9822B", "#8A94A6"
rng = np.random.default_rng(2026)

def state_to_bloch(amps):
    a, b = complex(amps[0]), complex(amps[1])
    n = np.sqrt(abs(a)**2 + abs(b)**2); a, b = a/n, b/n
    return np.array([2*(np.conj(a)*b).real, 2*(np.conj(a)*b).imag, abs(a)**2 - abs(b)**2])

def plot_bloch(amps_list, titles=None):
    """Kübit durumlarını yan yana Bloch küresinde çizer (1. haftadaki fonksiyon)."""
    if np.ndim(amps_list) == 1: amps_list = [amps_list]
    k = len(amps_list); fig = plt.figure(figsize=(3.6*k, 3.8))
    for i, amps in enumerate(amps_list):
        ax = fig.add_subplot(1, k, i+1, projection="3d"); ax.set_box_aspect((1,1,1), zoom=1.3); ax.computed_zorder = False
        u, v = np.linspace(0, 2*np.pi, 50), np.linspace(0, np.pi, 25)
        ax.plot_surface(np.outer(np.cos(u), np.sin(v)), np.outer(np.sin(u), np.sin(v)), np.outer(np.ones_like(u), np.cos(v)),
                        color="#EEF2F8", alpha=0.25, linewidth=0, shade=False)
        t = np.linspace(0, 2*np.pi, 200)
        ax.plot(np.cos(t), np.sin(t), 0, color=GRAY, lw=0.8); ax.plot(np.cos(t), 0*t, np.sin(t), color=GRAY, lw=0.5); ax.plot(0*t, np.cos(t), np.sin(t), color=GRAY, lw=0.5)
        for d in [(1,0,0), (0,1,0), (0,0,1)]: ax.plot([-d[0], d[0]], [-d[1], d[1]], [-d[2], d[2]], color=GRAY, lw=0.7, ls="--")
        for p, s in [((0,0,1.22),"|0⟩ (z)"), ((0,0,-1.25),"|1⟩"), ((1.42,0,0),"|+⟩ (x)"), ((-1.32,0,0),"|−⟩"), ((0,1.32,0),"|+i⟩ (y)"), ((0,-1.32,0),"|−i⟩")]:
            ax.text(*p, s, ha="center", va="center", fontsize=9.5, color=NAVY)
        x, y, z = state_to_bloch(amps)
        ax.plot([0, x], [0, y], [0, z], color=BLUE, lw=3); ax.scatter([x], [y], [z], color=BLUE, s=60, depthshade=False)
        ax.set_xlim(-1.1, 1.1); ax.set_ylim(-1.1, 1.1); ax.set_zlim(-1.1, 1.1); ax.view_init(elev=18, azim=30); ax.set_axis_off()
        if titles: ax.set_title(titles[i], fontsize=11, color=NAVY)
    plt.show()
print("hazır")

---
## A · Karmaşık sayılar
Bir karmaşık sayı `z = a + bi`, 2 boyutlu düzlemde bir noktadır. Python'da sanal birim `j` ile yazılır: `3 + 4j`.

| İşlem | Formül | Python |
|---|---|---|
| Gerçek / sanal kısım | a, b | `z.real`, `z.imag` |
| Eşlenik | z* = a − bi | `np.conj(z)` |
| Büyüklük | \|z\| = √(a² + b²) | `abs(z)` |
| Büyüklüğün karesi | \|z\|² = z·z* | `abs(z)**2` |
| Açı (faz) | φ = atan2(b, a) | `np.angle(z)` |
| Kutupsal yazım | z = r·e^(iφ) | `r * np.exp(1j*phi)` |

**Kuantumda neden önemli?** Olasılık = genliğin **büyüklüğünün karesi**. Genliğin açısı (faz) ise ölçümü doğrudan etkilemez ama kapılar arasında bilgi taşır.

In [ ]:
z = 3 + 4j
print("z        =", z)
print("eşlenik  =", np.conj(z))
print("|z|      =", abs(z))
print("|z|²     =", abs(z)**2, "  = z * z* =", (z*np.conj(z)).real)
print("faz (°)  =", np.degrees(np.angle(z)))

r, phi = abs(z), np.angle(z)
print("kutupsaldan geri:", r*np.exp(1j*phi))

In [ ]:
# Faz çarpanı e^(iφ): büyüklüğü her zaman 1
for deg in [0, 45, 90, 180, 270]:
    w = np.exp(1j*np.radians(deg))
    print(f"φ = {deg:3d}°  ->  e^(iφ) = {w:.3f}   |e^(iφ)| = {abs(w):.3f}")

---
## B · Vektörler ve iç çarpım
Kuantumda durumlar **sütun vektörleridir**. İki vektörün iç çarpımı:
$$\langle a|b\rangle = \sum_i a_i^* \, b_i \qquad \text{NumPy: } \texttt{np.vdot(a, b)}$$
(`np.vdot` birinci vektörün eşleniğini otomatik alır; `np.dot` almaz!)

**Genel olasılık kuralı:** q durumundaki bir kübitin, hedef durum φ olarak bulunma olasılığı
$$P = |\langle \varphi | q \rangle|^2$$
1\. haftadaki P(0) = |α|² kuralı, bunun φ = |0⟩ için özel hâlidir.

In [ ]:
ket0 = np.array([1, 0], dtype=complex)
ket1 = np.array([0, 1], dtype=complex)
plus = np.array([1, 1], dtype=complex) / np.sqrt(2)
minus = np.array([1, -1], dtype=complex) / np.sqrt(2)
q = np.array([0.6, 0.8], dtype=complex)

print("⟨0|q⟩  =", np.vdot(ket0, q), "-> P =", abs(np.vdot(ket0, q))**2)
print("⟨+|q⟩  =", np.vdot(plus, q), "-> P =", abs(np.vdot(plus, q))**2)
print("⟨+|−⟩  =", np.vdot(plus, minus), " (dik vektörler: iç çarpım 0)")
print("‖q‖    =", np.linalg.norm(q), " = sqrt(⟨q|q⟩) =", np.sqrt(np.vdot(q, q).real))

# np.dot ile np.vdot farkı
a = np.array([1j, 0]); print("np.dot(a,a) =", np.dot(a, a), "  np.vdot(a,a) =", np.vdot(a, a), " <- doğrusu")

---
## C · Matrisler = kapılar
Bir kapı, vektörü değiştiren bir **2×2 matristir**: `q_yeni = U @ q`. Matris çarpımı fonksiyon bileşimidir.

⚠️ **Sıra kuralı:** Devrede önce H sonra X uygulanıyorsa matematikte **X · H** yazılır (sağdaki önce uygulanır).

In [ ]:
I = np.eye(2, dtype=complex)
X = np.array([[0, 1], [1, 0]], dtype=complex)
Y = np.array([[0, -1j], [1j, 0]], dtype=complex)
Z = np.array([[1, 0], [0, -1]], dtype=complex)
H = np.array([[1, 1], [1, -1]], dtype=complex) / np.sqrt(2)
S = np.array([[1, 0], [0, 1j]], dtype=complex)
T = np.array([[1, 0], [0, np.exp(1j*np.pi/4)]], dtype=complex)
GATES = {"I": I, "X": X, "Y": Y, "Z": Z, "H": H, "S": S, "T": T}

print("H @ [0.6, 0.8] =", H @ q)
print("X·Z =\n", X @ Z, "\nZ·X =\n", Z @ X, "\n-> sıra önemli: XZ ≠ ZX")

# Devre: önce H, sonra X
qc = QuantumCircuit(1); qc.h(0); qc.x(0)
print("Qiskit:", Statevector.from_instruction(qc).data)
print("X@H@|0⟩:", X @ H @ ket0, " (doğru sıra)")
print("H@X@|0⟩:", H @ X @ ket0, " (yanlış sıra)")

### Üniterlik: her kapının sağlaması gereken kural
$$U^\dagger U = I \qquad (U^\dagger = \text{eşlenik transpoz} = \texttt{U.conj().T})$$
Üniter matrisler vektörün uzunluğunu korur, yani **olasılıkların toplamı hep 1 kalır**. Ayrıca her kapı **geri alınabilir**: tersi $U^\dagger$'dir.

In [ ]:
def dagger(M): return M.conj().T

for name, U in GATES.items():
    print(f"{name}: U†U = I ?  {np.allclose(dagger(U) @ U, I)}")

bad = np.array([[1, 1], [0, 1]], dtype=complex)
print("\n[[1,1],[0,1]] üniter mi?", np.allclose(dagger(bad) @ bad, I))
v = bad @ plus
print("Uygulanınca olasılık toplamı:", np.sum(abs(v)**2), " <- 1 değil, geçerli kapı olamaz!")
print("H'nin tersi kendisidir: H·H =\n", H @ H)

---
## D · Faz: global ve göreli
- **Global faz:** Vektörün tamamını aynı e^(iφ) ile çarpmak → olasılıklar ve Bloch noktası **değişmez**. Bu iki vektör aynı kuantum durumudur.
- **Göreli faz:** Sadece bir genliği e^(iφ) ile çarpmak → olasılıklar değişmez ama Bloch noktası **ekvator etrafında döner**. Bu gerçek bir farktır ve sonraki kapılar bunu görür.

In [ ]:
q = np.array([0.6, 0.8], dtype=complex)
print("q      olasılıklar:", abs(q)**2, " Bloch:", state_to_bloch(q))
print("i·q    olasılıklar:", abs(1j*q)**2, " Bloch:", state_to_bloch(1j*q), " <- aynı")

states = [np.array([1, np.exp(1j*p)])/np.sqrt(2) for p in [0, np.pi/2, np.pi, 3*np.pi/2]]
for s_, p in zip(states, ["0", "π/2", "π", "3π/2"]):
    print(f"[1, e^(i·{p})]/√2  olasılıklar = {abs(s_)**2}")
plot_bloch(states, ["φ=0 (|+⟩)", "φ=π/2 (|+i⟩)", "φ=π (|−⟩)", "φ=3π/2 (|−i⟩)"])

In [ ]:
# Göreli faz ölçümle görülmez ama H kapısından sonra görünür hâle gelir:
for name, s_ in [("|+⟩", plus), ("|−⟩", minus)]:
    print(f"{name}: ölçüm olasılıkları {abs(s_)**2}   H uygulandıktan sonra {abs(H @ s_)**2}")

---
## E · Tensör çarpımı: kübitleri birleştirmek
İki kübitin ortak durumu, vektörlerinin **Kronecker (tensör) çarpımıdır**: `np.kron(q1, q0)`.

| İndeks | Bitler (q₁q₀) | Genlik |
|---|---|---|
| 0 | 00 | a₀·b₀ |
| 1 | 01 | a₀·b₁ |
| 2 | 10 | a₁·b₀ |
| 3 | 11 | a₁·b₁ |

**Bu derste Qiskit sırasını kullanıyoruz:** `np.kron` içinde **en soldaki çarpan en yüksek numaralı kübit**tir (q₁ ⊗ q₀). Böylece indeksin ikili yazılışı Qiskit'in sonuç stringiyle aynı olur.

In [ ]:
def ket(bits):
    """'01' gibi bir stringden durum vektörü üretir (soldaki karakter en yüksek numaralı kübit)."""
    v = np.array([1], dtype=complex)
    for b in bits:
        v = np.kron(v, ket0 if b == "0" else ket1)
    return v

print("ket('01') =", ket("01"), " -> indeks", np.argmax(abs(ket("01"))))
q1, q0 = np.array([0.6, 0.8]), np.array([1, 0])
print("[0.6,0.8] ⊗ [1,0] =", np.kron(q1, q0))
print("|0⟩ ⊗ |+⟩ =", np.kron(ket0, plus))

# 2 kübitli kapılar: X yalnız q0'a -> I ⊗ X,   H yalnız q1'e -> H ⊗ I
print("(I⊗X)|00⟩ =", np.kron(I, X) @ ket("00"), " = |01⟩")
print("(X⊗I)|00⟩ =", np.kron(X, I) @ ket("00"), " = |10⟩")

qc = QuantumCircuit(2); qc.x(0)
print("Qiskit, x(0):", Statevector.from_instruction(qc).data, " -> aynı (I⊗X)")

### Dolanık durum çarpana ayrılamaz
Bell durumu `[0.707, 0, 0, 0.707]` iki ayrı kübit vektörünün tensör çarpımı olarak **yazılamaz**:
a₀b₀ ≠ 0 ve a₁b₁ ≠ 0 ise tüm a'lar ve b'ler sıfırdan farklıdır; ama o zaman a₀b₁ = 0 olamaz. Çelişki! Bu yüzden dolanık kübitlerin "kendi ayrı vektörü" yoktur.

In [ ]:
def is_product_state(psi, tol=1e-9):
    """2 kübitli durum çarpım durumu mu? 2x2 matrise yeniden şekillendirip rankına bakarız."""
    return np.linalg.matrix_rank(psi.reshape(2, 2), tol=tol) == 1

bell = np.array([1, 0, 0, 1]) / np.sqrt(2)
print("|0⟩⊗|+⟩ çarpım durumu mu?", is_product_state(np.kron(ket0, plus)))
print("Bell durumu çarpım durumu mu?", is_product_state(bell))

---
## F · MiniSim: kendi kuantum simülatörümüz
Şimdiye kadar öğrendiğimiz her şeyi tek bir sınıfta birleştiriyoruz:
- **Durum:** 2ⁿ elemanlı karmaşık vektör, başlangıç |00…0⟩
- **Tek kübit kapısı:** `I ⊗ … ⊗ U ⊗ … ⊗ I` tam matrisini kurup çarp
- **CNOT:** kontrol biti 1 olan indekslerde hedef bitini çevir (genliklerin yerini değiştir)
- **Ölçüm:** olasılıklar = |genlik|², örnekleme ile sayımlar

In [ ]:
class MiniSim:
    """Eğitim amaçlı durum vektörü simülatörü. Bit sırası Qiskit ile aynıdır (q0 en sağda)."""

    def __init__(self, n):
        self.n = n
        self.state = np.zeros(2**n, dtype=complex)
        self.state[0] = 1.0

    def _full_matrix(self, U, target):
        # Kron zincirinin en solu en yüksek numaralı kübit: q_{n-1} ⊗ ... ⊗ q_0
        M = np.array([[1]], dtype=complex)
        for k in reversed(range(self.n)):
            M = np.kron(M, U if k == target else I)
        return M

    def apply_1q(self, U, target):
        self.state = self._full_matrix(U, target) @ self.state
        return self

    def apply_cx(self, control, target):
        new = self.state.copy()
        for i in range(2**self.n):
            if (i >> control) & 1:              # kontrol biti 1 mi?
                j = i ^ (1 << target)           # hedef bitini çevir
                new[j] = self.state[i]
        self.state = new
        return self

    def probabilities(self):
        return np.abs(self.state)**2

    def sample(self, shots=1000, seed=None):
        r = np.random.default_rng(seed)
        idx = r.choice(2**self.n, size=shots, p=self.probabilities())
        vals, cnt = np.unique(idx, return_counts=True)
        return {format(int(v), f"0{self.n}b"): int(c) for v, c in zip(vals, cnt)}

    def prob_qubit_one(self, q):
        """Tek bir kübitin 1 okunma olasılığı (marjinal olasılık)."""
        return sum(p for i, p in enumerate(self.probabilities()) if (i >> q) & 1)

# Bell durumu
sim = MiniSim(2).apply_1q(H, 0).apply_cx(0, 1)
print("MiniSim durum :", sim.state)
print("Sayımlar      :", sim.sample(1000, seed=1))
print("P(q0 = 1)     :", sim.prob_qubit_one(0))

In [ ]:
# Qiskit ile karşılaştırma: aynı devre, aynı vektör olmalı
qc = QuantumCircuit(2); qc.h(0); qc.cx(0, 1)
print("Qiskit :", Statevector.from_instruction(qc).data)
print("Eşit mi?", np.allclose(Statevector.from_instruction(qc).data, sim.state))

# 3 kübit GHZ
ghz = MiniSim(3).apply_1q(H, 0).apply_cx(0, 1).apply_cx(1, 2)
print("\nGHZ sayımları:", ghz.sample(1000, seed=2))

### Rastgele devre testi (birim test mantığı)
MiniSim'i 200 rastgele devrede Qiskit ile karşılaştırıyoruz. Yazılımda nasıl test yazıyorsak, simülatörümüzü de öyle doğruluyoruz.

In [ ]:
def random_test(n=3, depth=12, trials=200):
    names = ["H", "X", "Y", "Z", "S", "T"]
    for _ in range(trials):
        ms, qc = MiniSim(n), QuantumCircuit(n)
        for _ in range(depth):
            if rng.random() < 0.3:
                c, t = rng.choice(n, 2, replace=False)
                ms.apply_cx(int(c), int(t)); qc.cx(int(c), int(t))
            else:
                g, t = rng.choice(names), int(rng.integers(n))
                ms.apply_1q(GATES[g], t); getattr(qc, g.lower())(t)
        if not np.allclose(ms.state, Statevector.from_instruction(qc).data):
            return False
    return True

print("200 rastgele devrede MiniSim == Qiskit ?", random_test())

### Dikkat: tam matris yöntemi büyümez
Tam matris 2ⁿ × 2ⁿ = 4ⁿ eleman içerir. 12 kübitte 16 milyon, 20 kübitte 10¹² eleman! Gerçek simülatörler matrisi kurmaz, vektörü yeniden şekillendirip (`reshape`) sadece ilgili ekseni çarpar. İsteyenler için verimli sürüm:

In [ ]:
def apply_1q_fast(state, U, target, n):
    psi = state.reshape([2]*n)                  # eksen k  <->  kübit (n-1-k)
    axis = n - 1 - target
    psi = np.tensordot(U, psi, axes=([1], [axis]))
    psi = np.moveaxis(psi, 0, axis)
    return psi.reshape(-1)

n = 3; s0 = MiniSim(n).apply_1q(H, 1).state
s1 = apply_1q_fast(ket("000"), H, 1, n)
print("Hızlı yöntem aynı sonucu veriyor mu?", np.allclose(s0, s1))

---
## G · Beklenen değer ⟨Z⟩
Kuantum makine öğrenmesinde modelin çıktısı genellikle bir **beklenen değerdir**:
$$\langle Z \rangle = \langle q | Z | q \rangle = P(0) - P(1)$$
Bu, Bloch küresindeki **z koordinatının ta kendisidir**. Benzer şekilde ⟨X⟩ = x koordinatı, ⟨Y⟩ = y koordinatıdır.

In [ ]:
def expval(U, psi): return np.vdot(psi, U @ psi).real

q = np.array([0.6, 0.8], dtype=complex)
print("⟨Z⟩ =", expval(Z, q), "  P(0)-P(1) =", 0.36 - 0.64)
print("⟨X⟩ =", expval(X, q), "  ⟨Y⟩ =", expval(Y, q))
print("Bloch (x, y, z) =", state_to_bloch(q))

---
## H · Alıştırmalar
`# TODO` yerlerini doldurun; `assert` satırları geçerse çözüm doğrudur.

### Alıştırma 1 · Kutupsal gösterim
`polar(z)` fonksiyonu `(r, derece)` döndürsün.

In [ ]:
def polar(z):
    # TODO
    pass

assert np.allclose(polar(3+4j), (5, 53.130102))
assert np.allclose(polar(1j), (1, 90))
assert np.allclose(polar(-1), (1, 180))
print("Alıştırma 1 ✓")

### Alıştırma 2 · Aynı durum mu? (global faz)
`same_state(a, b)`: iki normalize vektör global faz farkı dışında aynıysa `True` döndürsün. İpucu: |⟨a|b⟩| = 1.

In [ ]:
def same_state(a, b):
    # TODO
    pass

assert same_state([0.6, 0.8], [0.6j, 0.8j])
assert same_state([1, 0], [-1, 0])
assert not same_state(plus, minus)
print("Alıştırma 2 ✓")

### Alıştırma 3 · Üniterlik testi
`is_unitary(M)` fonksiyonunu yazın. Sonra `Rx(θ) = [[cos(θ/2), −i·sin(θ/2)], [−i·sin(θ/2), cos(θ/2)]]` matrisinin θ = 0.7 için üniter olduğunu doğrulayın.

In [ ]:
def is_unitary(M):
    # TODO
    pass

th = 0.7
Rx = np.array([[np.cos(th/2), -1j*np.sin(th/2)], [-1j*np.sin(th/2), np.cos(th/2)]])
assert all(is_unitary(G) for G in GATES.values())
assert is_unitary(Rx)
assert not is_unitary(np.array([[1, 1], [0, 1]]))
print("Alıştırma 3 ✓")

### Alıştırma 4 · Genel olasılık
`overlap_prob(phi, psi)` = |⟨φ|ψ⟩|². q = [0.6, 0.8] için |+⟩ ve |−⟩ olasılıklarını hesaplayın; toplamlarının 1 olduğunu gösterin.

In [ ]:
def overlap_prob(phi, psi):
    # TODO
    pass

q = np.array([0.6, 0.8])
p_plus, p_minus = overlap_prob(plus, q), overlap_prob(minus, q)
print(p_plus, p_minus)
assert np.isclose(p_plus, 0.98) and np.isclose(p_plus + p_minus, 1)
print("Alıştırma 4 ✓")

### Alıştırma 5 · CZ kapısı
MiniSim'e `apply_cz(c, t)` metodunu ekleyin: iki kübit de 1 olan indekslerde genliği −1 ile çarpar. Qiskit'in `cz` kapısıyla karşılaştırın.

In [ ]:
def apply_cz(self, c, t):
    # TODO
    return self
MiniSim.apply_cz = apply_cz

ms = MiniSim(2).apply_1q(H, 0).apply_1q(H, 1).apply_cz(0, 1)
qc = QuantumCircuit(2); qc.h([0, 1]); qc.cz(0, 1)
assert np.allclose(ms.state, Statevector.from_instruction(qc).data)
print("Alıştırma 5 ✓", ms.state)

### Alıştırma 6 · Üç CNOT ile SWAP
`CX(0,1) → CX(1,0) → CX(0,1)` dizisinin iki kübitin değerlerini takas ettiğini (SWAP) MiniSim ile gösterin: başlangıç `[0.6,0.8] ⊗ [1,0]` iken sonuç `[1,0] ⊗ [0.6,0.8]` olmalı.

In [ ]:
ms = MiniSim(2)
ms.state = np.kron([0.6, 0.8], [1, 0]).astype(complex)   # q1 = [0.6,0.8], q0 = [1,0]
# TODO: üç CNOT uygulayın

assert np.allclose(ms.state, np.kron([1, 0], [0.6, 0.8]))
print("Alıştırma 6 ✓", ms.state)

### Alıştırma 7 · Sayımlardan ⟨Z⟩ tahmini
`expz_from_counts(counts)` tek kübitlik sayımlardan ⟨Z⟩ = (N₀ − N₁)/N tahmin etsin. Qiskit'te `ry(θ)` ile [0.6, 0.8] durumunu hazırlayın (θ = 2·arccos(0.6)), 4000 shot çalıştırın ve tahmini gerçek değer −0.28 ile karşılaştırın.

In [ ]:
def expz_from_counts(counts):
    # TODO
    pass

theta = 2*np.arccos(0.6)
qc = QuantumCircuit(1, 1); qc.ry(theta, 0); qc.measure(0, 0)
aer = AerSimulator(seed_simulator=7)
counts = aer.run(transpile(qc, aer), shots=4000).result().get_counts()
est = expz_from_counts(counts)
print("tahmin:", est, " gerçek: -0.28")
assert abs(est + 0.28) < 0.05

---
### Haftanın özeti
- Olasılık = |genlik|²; genlik karmaşık olabilir (`abs(z)**2`)
- İç çarpım `np.vdot` → genel olasılık |⟨φ|q⟩|²
- Kapı = üniter matris (U†U = I); devre soldan sağa, matematik sağdan sola
- Global faz önemsiz, **göreli faz önemli** (ekvator etrafında dönüş)
- Tensör çarpımı `np.kron(q1, q0)`: indeksin ikili yazılışı = Qiskit sonuç stringi
- MiniSim ile ~40 satırda çalışan, Qiskit ile doğrulanmış bir simülatör yazdık

**Gelecek hafta:** Kübit, ölçüm ve Bloch küresi derinlemesine: farklı bazlarda ölçüm, shot istatistiği ve Bloch küresi üzerinde durum okuma.